Ở notebook 01 - nhóm thực hiện khai phá dữ liệu, tiền xử lý dữ liệu ở `data/raw/train.csv`. Đó là file dữ liệu gốc, ban đầu

### 1. import thư viện và load dữ liệu

In [23]:
import pandas as pd
import numpy as np 

df = pd.read_csv("data/raw/train.csv")
df.shape

(1460, 81)

In [24]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [25]:
df.isna()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
1,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
2,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
3,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
4,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False
1456,False,False,False,False,False,False,True,False,False,False,...,False,True,False,True,False,False,False,False,False,False
1457,False,False,False,False,False,False,True,False,False,False,...,False,True,False,False,False,False,False,False,False,False
1458,False,False,False,False,False,False,True,False,False,False,...,False,True,True,True,False,False,False,False,False,False


In [26]:
df.isna().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

### 2. EDA - khai phá dữ liệu

In [27]:

missing = (df.isna().sum()
             .loc[lambda s: s > 0]
             .sort_values(ascending=False)
             .to_frame("so_luong"))
missing["ti_le_(%)"] = (missing["so_luong"] / len(df) * 100).round(2)
print(missing)

              so_luong  ti_le_(%)
PoolQC            1453      99.52
MiscFeature       1406      96.30
Alley             1369      93.77
Fence             1179      80.75
MasVnrType         872      59.73
FireplaceQu        690      47.26
LotFrontage        259      17.74
GarageType          81       5.55
GarageYrBlt         81       5.55
GarageFinish        81       5.55
GarageQual          81       5.55
GarageCond          81       5.55
BsmtExposure        38       2.60
BsmtFinType2        38       2.60
BsmtQual            37       2.53
BsmtCond            37       2.53
BsmtFinType1        37       2.53
MasVnrArea           8       0.55
Electrical           1       0.07


In [28]:
print(df["SalePrice"].describe(), "\nSkewness:", df["SalePrice"].skew())

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64 
Skewness: 1.8828757597682129


### 3. Preprocessing

In [29]:
df = df.drop(columns = "Id")
df["MSSubClass"] = df["MSSubClass"].astype(str)
print("So dong trung: ", df.duplicated().sum())

So dong trung:  0


In [30]:
from sklearn.model_selection import train_test_split

df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))]
X = df.drop(columns ="SalePrice")
Y = np.log1p(df["SalePrice"])
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

In [31]:
from pathlib import Path

path_out = Path("data/processed")

X_train.to_csv(path_out / "X_train.csv", index=False)
X_val.to_csv(path_out / "X_val.csv", index=False)
Y_train.to_csv(path_out / "Y_train.csv", index=False)
Y_val.to_csv(path_out / "Y_val.csv", index=False)